# Feature Transformation: Scaling, Encoding & Normalization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/03-feature-engineering/03_feature_transformation.ipynb)

## Objectives
- Learn scaling techniques (StandardScaler, MinMaxScaler)
- Master categorical encoding methods
- Understand normalization and standardization
- Apply log and Box-Cox transformations
- Choose appropriate transformations for different algorithms

## 1. Why Transform Features?

**Benefits of transformation:**
- ✅ Improve model convergence (gradient descent)
- ✅ Prevent features with large scales from dominating
- ✅ Make non-linear patterns linear
- ✅ Handle categorical variables
- ✅ Reduce impact of outliers
- ✅ Meet algorithm assumptions

**When NOT needed:**
- Tree-based models (RF, XGBoost) are scale-invariant
- When features are already in comparable ranges

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer,
    OneHotEncoder, LabelEncoder, OrdinalEncoder
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import scipy.stats as stats

np.random.seed(42)
sns.set_theme()

print("✅ Libraries loaded")

## 2. Create Sample Dataset with Mixed Features

In [ ]:
# Create dataset with varying scales and distributions
np.random.seed(42)
n_samples = 1000

df = pd.DataFrame({
    'age': np.random.randint(18, 80, n_samples),                           # 18-80 range
    'income': np.random.normal(50000, 20000, n_samples),                   # Mean 50k, std 20k
    'credit_score': np.random.randint(300, 850, n_samples),                # 300-850 range
    'years_employed': np.random.exponential(8, n_samples),                 # Skewed right
    'debt': np.random.lognormal(10, 1, n_samples),                         # Log-normal (positive skew)
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
    'employment_type': np.random.choice(['Full-time', 'Part-time', 'Self-employed'], n_samples),
    'loan_approved': np.random.choice([0, 1], n_samples, p=[0.6, 0.4])
})

print(f"Dataset shape: {df.shape}")
print(f"\nDescriptive Statistics:")
print(df.describe())
print(f"\nData types:")
print(df.dtypes)

## 3. Visualize Original Features

See the distribution differences

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

numeric_features = ['age', 'income', 'credit_score', 'years_employed', 'debt']

for idx, feature in enumerate(numeric_features):
    axes[idx].hist(df[feature], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{feature}\nRange: {df[feature].min():.1f} to {df[feature].max():.1f}', fontsize=10)
    axes[idx].set_ylabel('Frequency')

plt.suptitle('Original Feature Distributions (Different Scales)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: Features have vastly different ranges and distributions!")

## 4. Scaling Techniques for Numeric Features

### 4.1 StandardScaler (Z-score normalization)

# StandardScaler: (X - mean) / std
scaler_standard = StandardScaler()
df_standard = df.copy()
df_standard[numeric_features] = scaler_standard.fit_transform(df[numeric_features])

print("📊 StandardScaler Results:")
print(df_standard[numeric_features].describe())
print(f"\nMean (should be ≈0): {df_standard[numeric_features].mean().round(3).tolist()}")
print(f"Std (should be ≈1): {df_standard[numeric_features].std().round(3).tolist()}")

# MinMaxScaler: (X - min) / (max - min) -> [0, 1]
scaler_minmax = MinMaxScaler()
df_minmax = df.copy()
df_minmax[numeric_features] = scaler_minmax.fit_transform(df[numeric_features])

print("📊 MinMaxScaler Results:")
print(df_minmax[numeric_features].describe())
print(f"\nMin (should be 0): {df_minmax[numeric_features].min().round(3).tolist()}")
print(f"Max (should be 1): {df_minmax[numeric_features].max().round(3).tolist()}")

# RobustScaler: (X - median) / IQR -> handles outliers well
scaler_robust = RobustScaler()
df_robust = df.copy()
df_robust[numeric_features] = scaler_robust.fit_transform(df[numeric_features])

print("📊 RobustScaler Results:")
print(df_robust[numeric_features].describe())
print(f"\nMedian (should be ≈0): {df_robust[numeric_features].median().round(3).tolist()}")

# Compare scaling methods on single feature
feature_to_compare = 'income'

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Original
axes[0, 0].hist(df[feature_to_compare], bins=30, edgecolor='black', alpha=0.7, color='blue')
axes[0, 0].set_title('Original Income')
axes[0, 0].set_ylabel('Frequency')

# StandardScaler
axes[0, 1].hist(df_standard[feature_to_compare], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title('StandardScaler')
axes[0, 1].set_ylabel('Frequency')

# MinMaxScaler
axes[1, 0].hist(df_minmax[feature_to_compare], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_title('MinMaxScaler (0-1)')
axes[1, 0].set_ylabel('Frequency')

# RobustScaler
axes[1, 1].hist(df_robust[feature_to_compare], bins=30, edgecolor='black', alpha=0.7, color='red')
axes[1, 1].set_title('RobustScaler')
axes[1, 1].set_ylabel('Frequency')

plt.suptitle('Comparison of Scaling Methods', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1 One-Hot Encoding (for nominal categories)
# For features with no ordinal relationship

df_encoded = df.copy()

# One-Hot Encoding
encoder_onehot = OneHotEncoder(sparse_output=False, drop='first') # drop='first' to avoid multicollinearity
employment_encoded = encoder_onehot.fit_transform(df[['employment_type']])
employment_df = pd.DataFrame(
    employment_encoded,
    columns=[f'employment_{cat}' for cat in encoder_onehot.get_feature_names_out()]
)
df_encoded = pd.concat([df_encoded, employment_df], axis=1)

print("📊 One-Hot Encoding (Employment Type):")
print(f"Original column: employment_type")
print(f"Encoded columns: {list(employment_df.columns)}")
print(f"\nSample:")
print(df[['employment_type']].head(10))
print(df_encoded[[c for c in df_encoded.columns if 'employment' in c]].head(10))

# Label Encoding for ordinal features
education_mapping = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
df_encoded['education_encoded'] = df_encoded['education'].map(education_mapping)

print("📊 Label Encoding (Education - Ordinal):")
print(f"Original -> Encoded mapping:")
for orig, encoded in education_mapping.items():
    print(f"  {orig:12} -> {encoded}")

print(f"\nSample comparison:")
print(df_encoded[['education', 'education_encoded']].head(10))

# Target Encoding: replace category with mean target value
target_encoding = df.groupby('employment_type')['loan_approved'].mean()
df_encoded['employment_target'] = df_encoded['employment_type'].map(target_encoding)

print("📊 Target Encoding (Employment Type):")
print(f"Mean loan approval rate by employment type:")
print(target_encoding)

print(f"\nSample comparison:")
print(df_encoded[['employment_type', 'employment_target', 'loan_approved']].head(10))

### 6.1 Log Transformation
# For right-skewed data (like debt)

df_log = df.copy()
df_log['debt_log'] = np.log1p(df['debt'])  # log1p handles 0 values
df_log['years_employed_log'] = np.log1p(df['years_employed'])

print("📊 Log Transformation:")
print(f"Original Debt - Skewness: {stats.skew(df['debt']):.3f}")
print(f"Log(Debt) - Skewness: {stats.skew(df_log['debt_log']):.3f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df['debt'], bins=30, edgecolor='black', alpha=0.7, color='blue')
axes[0, 0].set_title('Original Debt (Right-Skewed)')

axes[0, 1].hist(df_log['debt_log'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title('Log-Transformed Debt')

axes[1, 0].hist(df['years_employed'], bins=30, edgecolor='black', alpha=0.7, color='blue')
axes[1, 0].set_title('Original Years Employed (Right-Skewed)')

axes[1, 1].hist(df_log['years_employed_log'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_title('Log-Transformed Years Employed')

plt.suptitle('Log Transformation for Skewed Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Box-Cox finds optimal power transformation (only for positive values)
from scipy.stats import boxcox

# Apply Box-Cox to skewed features
debt_boxcox, debt_lambda = boxcox(df['debt'] + 1)  # +1 to ensure positive
years_boxcox, years_lambda = boxcox(df['years_employed'] + 0.1)

print("📊 Box-Cox Transformation:")
print(f"Debt - Lambda: {debt_lambda:.3f}, Original Skewness: {stats.skew(df['debt']):.3f}, Transformed Skewness: {stats.skew(debt_boxcox):.3f}")
print(f"Years Employed - Lambda: {years_lambda:.3f}, Original Skewness: {stats.skew(df['years_employed']):.3f}, Transformed Skewness: {stats.skew(years_boxcox):.3f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df['debt'], bins=30, edgecolor='black', alpha=0.7, color='red')
axes[0, 0].set_title('Original Debt')

axes[0, 1].hist(debt_boxcox, bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title(f'Box-Cox Debt (λ={debt_lambda:.3f})')

axes[1, 0].hist(df['years_employed'], bins=30, edgecolor='black', alpha=0.7, color='blue')
axes[1, 0].set_title('Original Years Employed')

axes[1, 1].hist(years_boxcox, bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_title(f'Box-Cox Years (λ={years_lambda:.3f})')

plt.suptitle('Box-Cox Transformation (Optimal Power)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# PowerTransformer can use Yeo-Johnson (handles negative values)
pt = PowerTransformer(method='yeo-johnson')
df_yeojohnson = pt.fit_transform(df[[' debt', 'years_employed', 'income']])

print("📊 Yeo-Johnson Transformation:")
print(f"Original Income - Skewness: {stats.skew(df['income']):.3f}")
print(f"Transformed Income - Skewness: {stats.skew(df_yeojohnson[:, 2]):.3f}")

# Prepare datasets for comparison
X_original = df[numeric_features].copy()
X_scaled = df_standard[numeric_features].copy()
X_minmax = df_minmax[numeric_features].copy()
X_robust = df_robust[numeric_features].copy()

y = df['loan_approved']

# Train-test split
X_train_orig, X_test_orig, y_train, y_test = train_test_split(X_original, y, test_size=0.2, random_state=42)
X_train_scaled, X_test_scaled, _, _ = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
X_train_minmax, X_test_minmax, _, _ = train_test_split(X_minmax, y, test_size=0.2, random_state=42)
X_train_robust, X_test_robust, _, _ = train_test_split(X_robust, y, test_size=0.2, random_state=42)

# Train Logistic Regression (sensitive to scaling)
results = {}

for name, X_train, X_test in [
    ('Original (no scaling)', X_train_orig, X_test_orig),
    ('StandardScaler', X_train_scaled, X_test_scaled),
    ('MinMaxScaler', X_train_minmax, X_test_minmax),
    ('RobustScaler', X_train_robust, X_test_robust)
]:
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_train, y_train)
    acc = accuracy_score(y_test, lr.predict(X_test))
    results[name] = acc

results_df = pd.DataFrame(list(results.items()), columns=['Method', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False)

print("📊 Impact on Logistic Regression (Scale-Sensitive):")
print(results_df.to_string(index=False))

# Visualization
plt.figure(figsize=(10, 6))
plt.bar(range(len(results_df)), results_df['Accuracy'], color=['green' if x > results_df['Accuracy'].max() - 0.01 else 'skyblue' for x in results_df['Accuracy']])
plt.xticks(range(len(results_df)), results_df['Method'], rotation=45, ha='right')
plt.ylabel('Accuracy')
plt.title('Impact of Feature Scaling on Model Performance')
for i, v in enumerate(results_df['Accuracy']):
    plt.text(i, v + 0.001, f'{v:.4f}', ha='center')
plt.tight_layout()
plt.show()

print("""\n
📋 TRANSFORMATION DECISION FRAMEWORK:

1. NUMERIC FEATURES - SCALING:
   ├─ StandardScaler (most common):
   │  ├─ Use for: Linear models (LR, SVM), Neural Networks, KNN
   │  ├─ Range: (-∞, ∞)
   │  └─ Good for: Normal distributions
   │
   ├─ MinMaxScaler:
   │  ├─ Use for: Neural Networks, When bounded [0,1] preferred
   │  ├─ Range: [0, 1]
   │  └─ Bad for: Outliers (stretches scale)
   │
   └─ RobustScaler:
      ├─ Use for: Data with outliers
      ├─ Uses: Median and IQR
      └─ Good for: Robust to outliers

2. SKEWED FEATURES:
   ├─ Log Transform:
   │  ├─ Use for: Right-skewed positive data
   │  └─ Example: Income, prices, counts
   │
   └─ Box-Cox or Yeo-Johnson:
      ├─ Use for: Automatic optimal transformation
      └─ Box-Cox: Positive values only

3. CATEGORICAL FEATURES:
   ├─ One-Hot Encoding:
   │  ├─ Use for: Nominal (no order) categories
   │  └─ Example: Color, employment_type
   │
   ├─ Label Encoding:
   │  ├─ Use for: Ordinal (has order) categories
   │  └─ Example: Education level, severity
   │
   └─ Target Encoding:
      ├─ Use for: High cardinality, want to capture target info
      └─ Caveat: Beware of overfitting

4. TREE-BASED MODELS:
   └─ No scaling needed! (XGBoost, Random Forest)
""")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define transformations
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, drop='first'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, ['employment_type', 'education'])
    ]
)

# Create full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Fit and evaluate
X_train, X_test, y_train, y_test = train_test_split(
    df[numeric_features + ['employment_type', 'education']], 
    df['loan_approved'], 
    test_size=0.2, 
    random_state=42
)

pipeline.fit(X_train, y_train)
acc = accuracy_score(y_test, pipeline.predict(X_test))

print(f"✅ Pipeline accuracy: {acc:.4f}")
print(f"\n✅ Automatic transformations applied to all features!")

print("""\n📚 KEY TAKEAWAYS:

✅ SCALING (Numeric):
   • StandardScaler: Default choice for most algorithms
   • MinMaxScaler: When bounded range [0,1] needed
   • RobustScaler: When outliers present

✅ TRANSFORMATIONS (Skewed):
   • Log: Right-skewed positive data
   • Box-Cox: Automatic optimal transformation

✅ ENCODING (Categorical):
   • One-Hot: Nominal features (no order)
   • Label: Ordinal features (has order)
   • Target: High cardinality with caution

✅ BEST PRACTICES:
   1. Fit transformers on TRAINING data only
   2. Apply same transformers to TEST data
   3. Tree models DON'T need scaling
   4. Use pipelines to prevent data leakage
   5. Always validate improvements on test set

❌ COMMON MISTAKES:
   1. Fitting on train+test combined (data leakage!)
   2. Forgetting to transform test data
   3. Over-transforming (sometimes simpler is better)
   4. Not documenting transformations
""")